# ray-parametric-form composite — cx1: build (NR, 3, 3) ray/triangle linear system by stacking columns

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ray-parametric-form`, `stack-vs-cat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ray-parametric-form"
DD_ATOM_IDS = ["ray-parametric-form", "stack-vs-cat"]
DD_SUBTOPICS = ["Geometry: Ray parametric form", "PyTorch: stack vs cat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Ray-triangle intersection reduces to the 3x3 linear system

  `[-D | B-A | C-A] @ [u, v, w] = O - A`

where `O, D` are the ray origin/direction and `A, B, C` are the triangle vertices. The left-hand-side matrix is built from THREE COLUMN VECTORS — each of shape `(3,)` per ray — that need to be assembled into a `(3, 3)` matrix. Across `NR` rays we want a `(NR, 3, 3)` batch.

This is the `stack-vs-cat` question in its purest form. We have three `(NR, 3)` column tensors and need a `(NR, 3, 3)` output. Same rank as the inputs means... NO — we need to INSERT a new axis of size 3 (one per column), so `t.stack(..., dim=-1)` is the right call. `cat(dim=-1)` would give `(NR, 9)`, wrong rank, silent shape bug.

The `ray-parametric-form` atom is exercised in the column construction itself: `-D` IS the direction column, scaled by `-1` — i.e. the parametric form `R(u) = O + u*D` rearranged so `D` appears on the LHS as a coefficient of the `u` unknown.

### Composite Exercise — build (NR, 3, 3) ray/triangle linear system by stacking columns

**Atoms exercised together**: `ray-parametric-form`, `stack-vs-cat`

Implement `cx1_build_linsys(rays, triangle)` that builds the `(NR, 3, 3)` linear-system matrix for ray-triangle intersection.

- `rays` has shape `(NR, 2, 3)` — `rays[r, 0]` is origin `O_r`, `rays[r, 1]` is direction `D_r`.
- `triangle` has shape `(3, 3)` — rows are vertices `A, B, C`.

1. **Construct three column vectors**, each of shape `(NR, 3)`:
   - `col0 = -D` (negated ray direction; this is the `ray-parametric-form` half — the LHS coefficient of the ray-parameter unknown).
   - `col1 = B - A` (constant across rays — broadcast it to `(NR, 3)`).
   - `col2 = C - A` (same).
2. **Stack** the three columns into a `(NR, 3, 3)` matrix. Use `t.stack(..., dim=-1)` — that INSERTS a new trailing axis of size 3 (one per column). Do NOT use `cat` (which would extend an existing axis and give the wrong rank).

Return the `(NR, 3, 3)` LHS matrix. The downstream solve `mat @ [u,v,w] = O - A` uses this exact shape contract.

In [ ]:
def cx1_build_linsys(rays, triangle):
    NR = rays.shape[0]
    # ray-parametric-form: the LHS coefficient of u in R(u)=O+u*D is -D after moving to LHS.
    D = rays[:, 1]                  # (NR, 3)
    A, B, C = triangle[0], triangle[1], triangle[2]
    col0 = -D                       # (NR, 3)
    # Broadcast the constant column vectors across NR rays.
    col1 = (B - A).expand(NR, 3)    # (NR, 3)
    col2 = (C - A).expand(NR, 3)    # (NR, 3)
    # stack-vs-cat: we need a NEW axis (3 columns) -> stack(dim=-1), not cat.
    return t.stack([col0, col1, col2], dim=-1)  # (NR, 3, 3)


<details><summary>Show solution — cx1</summary>

```python
def cx1_build_linsys(rays, triangle):
    NR = rays.shape[0]
    # ray-parametric-form: the LHS coefficient of u in R(u)=O+u*D is -D after moving to LHS.
    D = rays[:, 1]                  # (NR, 3)
    A, B, C = triangle[0], triangle[1], triangle[2]
    col0 = -D                       # (NR, 3)
    # Broadcast the constant column vectors across NR rays.
    col1 = (B - A).expand(NR, 3)    # (NR, 3)
    col2 = (C - A).expand(NR, 3)    # (NR, 3)
    # stack-vs-cat: we need a NEW axis (3 columns) -> stack(dim=-1), not cat.
    return t.stack([col0, col1, col2], dim=-1)  # (NR, 3, 3)
```

The `stack(dim=-1)` is doing the load-bearing work: each of `col0`, `col1`, `col2` has shape `(NR, 3)`; we want them to become columns 0, 1, 2 of an `(NR, 3, 3)` matrix. Inserting a NEW trailing axis is the textbook `stack` case. `cat(dim=-1)` would concatenate along the existing trailing axis and give `(NR, 9)` — wrong rank, fails the downstream solve.

Note that `-D` is the `ray-parametric-form` atom rearranged: starting from `R(u) = O + u*D` and asking `R(u) ∈ triangle`, you end up with `-u*D + v*(B-A) + w*(C-A) = O - A`, so the matrix has `-D` as its first column.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx1',
        'subtopics': ["Geometry: Ray parametric form", "PyTorch: stack vs cat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()